# Prompt Engineering Portfolio

This notebook demonstrates 11+ prompt engineering techniques with real examples for text classification, summarization, code generation, and data extraction. It compares GPT, Groq, and a local Ollama-based model when available.

## Setup and provider wrappers

The next cell loads environment variables and defines wrappers for OpenAI, Groq, and Ollama (if installed locally).

In [5]:
import os
import json
import shutil
import subprocess
import textwrap
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
OLLAMA_INSTALLED = shutil.which('ollama') is not None

print('OpenAI key available:', bool(OPENAI_API_KEY))
print('Groq key available:', bool(GROQ_API_KEY))
print('Ollama installed:', OLLAMA_INSTALLED)

OpenAI key available: True
Groq key available: True
Ollama installed: True


In [ ]:
def safe_text(response):
    """Extract text from various response formats."""
    if response is None:
        return ''
    if isinstance(response, str):
        return response.strip()
    try:
        if hasattr(response, 'output_text'):
            return response.output_text.strip()
        if hasattr(response, 'content') and isinstance(response.content, str):
            return response.content.strip()
        if hasattr(response, 'output'):
            pieces = []
            for item in response.output:
                if isinstance(item, dict) and item.get('type') == 'output_text':
                    for content in item.get('content', []):
                        pieces.append(content.get('text', ''))
            if pieces:
                return ''.join(pieces).strip()
    except Exception:
        pass
    try:
        return json.dumps(response, default=str, indent=2)
    except Exception:
        return str(response)

openai_client = None
groq_client = None

if OPENAI_API_KEY:
    try:
        from openai import OpenAI
        openai_client = OpenAI(api_key=OPENAI_API_KEY)
    except Exception as exc:
        print('OpenAI client import failed:', exc)

if GROQ_API_KEY:
    try:
        from groq import Groq
        groq_client = Groq(api_key=GROQ_API_KEY)
    except Exception as exc:
        print('Groq client import failed:', exc)

def call_openai(prompt, model='gpt-4o-mini', temperature=0.2):
    """Call OpenAI API."""
    if not openai_client:
        return 'OpenAI client not configured. Set OPENAI_API_KEY.'
    try:
        response = openai_client.chat.completions.create(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=temperature,
            max_tokens=400
        )
        return response.choices[0].message.content.strip()
    except Exception as exc:
        return f'OpenAI call failed: {exc}'

def call_groq(prompt, model='llama3-70b-8192', temperature=0.2):
    """Call Groq API."""
    if not groq_client:
        return 'Groq client not configured. Set GROQ_API_KEY.'
    try:
        response = groq_client.chat.completions.create(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=temperature,
            max_tokens=400
        )
        return response.choices[0].message.content.strip()
    except Exception as exc:
        return f'Groq call failed: {exc}'

def call_ollama(prompt, model='llama2'):
    """Call Ollama local model."""
    if not OLLAMA_INSTALLED:
        return 'Ollama not installed in this environment.'
    try:
        result = subprocess.run(
            ['ollama', 'run', model],
            input=prompt,
            capture_output=True,
            text=True,
            timeout=120
        )
        if result.returncode == 0:
            return result.stdout.strip()
        return f'Ollama error: {result.stderr.strip() or result.stdout.strip()}'
    except Exception as exc:
        return f'Ollama execution failed: {exc}'

providers = {
    'GPT': lambda prompt: call_openai(prompt, model='gpt-4o-mini'),
    'Groq': lambda prompt: call_groq(prompt, model='mixtral-8x7b-32768'),
    'Ollama': lambda prompt: call_ollama(prompt, model='llama2'),
}

def compare_providers(prompt, label):
    """Compare responses from multiple providers."""
    results = {provider: fn(prompt) for provider, fn in providers.items()}
    print(f'\\n--- {label} ---')
    for provider, output in results.items():
        print(f"[{provider}]\\n{output}\\n")
    return results

## Technique 1: Zero-shot sentiment classification

This technique provides a simple, direct instruction to classify text without examples.

In [7]:
zero_shot_prompt = textwrap.dedent('''
You are a sentiment classifier. For each review, return one of: Positive, Neutral, Negative.

Review 1: I loved the product, it works beautifully and arrived faster than expected.
Review 2: The app keeps crashing and customer support did not respond.
Review 3: The interface is okay, but the pricing is a little high.
''')

zero_shot_results = compare_providers(zero_shot_prompt, 'Zero-shot sentiment classification')
pd.DataFrame(zero_shot_results, index=['Response']).T

Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\SrushtiMoghe(CloudPl\AppData\Local\Programs\Python\Python314\Lib\threading.py", line 1082, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "c:\Users\SrushtiMoghe(CloudPl\AppData\Local\Programs\Python\Python314\Lib\threading.py", line 1024, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\SrushtiMoghe(CloudPl\AppData\Local\Programs\Python\Python314\Lib\subprocess.py", line 1613, in _readerthread
    buffer.append(fh.read())
                  ~~~~~~~^^
  File "c:\Users\SrushtiMoghe(CloudPl\AppData\Local\Programs\Python\Python314\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8f in position 541: c

\n--- Zero-shot sentiment classification ---
[GPT]\nOpenAI call failed: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}\n
[Groq]\nGroq call failed: Error code: 400 - {'error': {'message': 'The model `mixtral-8x7b-32768` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}\n
[Ollama]\nOllama execution failed: Command '['ollama', 'run', 'llama2']' timed out after 120 seconds\n


,Response
GPT,OpenAI call failed: Error code: 429 - {'error'...
Groq,Groq call failed: Error code: 400 - {'error': ...
Ollama,"Ollama execution failed: Command '['ollama', '..."


## Technique 2: Few-shot classification

Provide labeled examples to guide the model toward consistent classification.

In [8]:
few_shot_prompt = textwrap.dedent('''
Use the examples below to classify the new review as Positive, Neutral, or Negative.

Example 1: The new update improved performance and I am happy. -> Positive
Example 2: The product is fine but missing a few useful features. -> Neutral
Example 3: I am disappointed and will not buy again. -> Negative

Review: The features are powerful but the installation was frustrating.
''')

few_shot_results = compare_providers(few_shot_prompt, 'Few-shot sentiment classification')
pd.DataFrame(few_shot_results, index=['Response']).T

Exception in thread Thread-7 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\SrushtiMoghe(CloudPl\AppData\Local\Programs\Python\Python314\Lib\threading.py", line 1082, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "c:\Users\SrushtiMoghe(CloudPl\AppData\Local\Programs\Python\Python314\Lib\threading.py", line 1024, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\SrushtiMoghe(CloudPl\AppData\Local\Programs\Python\Python314\Lib\subprocess.py", line 1613, in _readerthread
    buffer.append(fh.read())
                  ~~~~~~~^^
  File "c:\Users\SrushtiMoghe(CloudPl\AppData\Local\Programs\Python\Python314\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8f in position 541: c

\n--- Few-shot sentiment classification ---
[GPT]\nOpenAI call failed: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}\n
[Groq]\nGroq call failed: Error code: 400 - {'error': {'message': 'The model `mixtral-8x7b-32768` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}\n
[Ollama]\nOllama execution failed: Command '['ollama', 'run', 'llama2']' timed out after 120 seconds\n


,Response
GPT,OpenAI call failed: Error code: 429 - {'error'...
Groq,Groq call failed: Error code: 400 - {'error': ...
Ollama,"Ollama execution failed: Command '['ollama', '..."


## Technique 3: Chain-of-thought reasoning

Encourage the model to reason step-by-step before answering.

In [ ]:
cot_prompt = textwrap.dedent('''
Solve the question with step-by-step reasoning and then give the final answer.

Question: If a store sold 120 units in March and then sold 25% more in April, how many units did it sell in April?
''')

cot_results = compare_providers(cot_prompt, 'Chain-of-thought reasoning')
pd.DataFrame(cot_results, index=['Response']).T

## Technique 4: Structured output for data extraction

Ask the model to return valid JSON with specific fields.

In [ ]:
invoice_text = textwrap.dedent('''
Invoice ID: 76432
Date: 2026-02-18
Vendor: Acme Widgets
Total: $1,250.00
Due: 2026-03-18
Items:
  - Widget A x3 @ $250.00
  - Widget B x1 @ $500.00
''')

structured_prompt = textwrap.dedent(f'''
Extract the invoice details from the text below and respond only with JSON.
The JSON must include: invoice_id, date, vendor, total, due_date, and items.
Each item should have: description, quantity, and price.

Text:
{invoice_text}
''')

structured_results = compare_providers(structured_prompt, 'Structured extraction (JSON)')
pd.DataFrame(structured_results, index=['Response']).T

## Technique 5: Role-playing summarization

Ask the model to summarize from the perspective of a role, such as a product manager.

In [ ]:
meeting_notes = textwrap.dedent('''
The customer wants a faster onboarding flow, clearer error messages, and better reporting dashboards.
Development is on track for mobile release, but team bandwidth is tight.
We need to prioritize stability before adding new integrations.
''')

role_play_prompt = textwrap.dedent(f'''
You are a product manager writing an executive summary.
Summarize the meeting notes below, focusing on customer needs, risks, and the next priority.

Meeting notes:
{meeting_notes}
''')

role_play_results = compare_providers(role_play_prompt, 'Role-playing summarization')
pd.DataFrame(role_play_results, index=['Response']).T

## Technique 6: Code generation with constraints

Request a function with clear behavior and formatting requirements.

In [ ]:
code_prompt = textwrap.dedent('''
Write a Python function named `validate_email` that checks whether an email address is valid.
The function should use a regular expression, return True/False, and include a concise docstring.
Do not include any example usage or explanation.
''')

code_results = compare_providers(code_prompt, 'Code generation')
pd.DataFrame(code_results, index=['Response']).T

## Technique 7: Prompt decomposition

Break a larger task into smaller subtasks before answering.

In [ ]:
decomposition_prompt = textwrap.dedent('''
You are designing a product launch checklist.
First list the key launch phases. Then, for each phase, provide the most important task.
Return the result as a numbered list with phase and task.
''')

decomposition_results = compare_providers(decomposition_prompt, 'Prompt decomposition')
pd.DataFrame(decomposition_results, index=['Response']).T

## Technique 8: Self-critique and improvement

Ask the model to analyze its own initial response and improve it.

In [ ]:
self_critique_prompt = textwrap.dedent('''
Write a short executive summary of the product launch risks in three sentences.
Then critique the summary and rewrite it to be more specific and actionable.
''')

self_critique_results = compare_providers(self_critique_prompt, 'Self-critique')
pd.DataFrame(self_critique_results, index=['Response']).T

## Technique 9: Negative examples and clarification

Show what you DON'T want to help avoid common mistakes.

In [ ]:
negative_example_prompt = textwrap.dedent('''
Generate a professional customer support response to a complaint about late delivery.

BAD example: "Your package is late. Too bad."
GOOD example: "We apologize for the delay. We have escalated your case and will investigate immediately."

Customer complaint: My order arrived 3 days late with no tracking updates.
''')

negative_results = compare_providers(negative_example_prompt, 'Negative examples')
pd.DataFrame(negative_results, index=['Response']).T

## Technique 10: Temperature and output control

Adjust model temperature to control creativity vs. consistency.

In [ ]:
temperature_prompt = textwrap.dedent('''
List three creative ideas for a mobile app feature that helps users save time.
''')

print('\\n--- Temperature test: Low (more consistent) ---')
temp_low = {
    'GPT': call_openai(temperature_prompt, temperature=0.1),
    'Groq': call_groq(temperature_prompt, temperature=0.1),
}
for provider, output in temp_low.items():
    print(f"[{provider}]\n{output}\n")

print('\n--- Temperature test: High (more creative) ---')
temp_high = {
    'GPT': call_openai(temperature_prompt, temperature=0.8),
    'Groq': call_groq(temperature_prompt, temperature=0.8),
}
for provider, output in temp_high.items():
    print(f"[{provider}]\\n{output}\\n")

## Technique 11: Retrieval-Augmented Generation (RAG)

Combine external knowledge with the language model to ground responses in specific documents or data.

In [ ]:
knowledge_base = textwrap.dedent('''
Company Policy Q&A:
Q: What is the return policy?
A: Returns are accepted within 30 days of purchase for a full refund.
Q: What payment methods do you accept?
A: We accept credit cards, PayPal, and Apple Pay.
''')

rag_prompt = textwrap.dedent(f'''
Using ONLY the information below, answer the customer question.

Knowledge Base:
{knowledge_base}

Customer Question: Can I return my order?
''')

rag_results = compare_providers(rag_prompt, 'Retrieval-Augmented Generation (RAG)')
pd.DataFrame(rag_results, index=['Response']).T

## Provider availability summary

In [ ]:
availability = {
    'Provider': ['GPT', 'Groq', 'Ollama'],
    'Available': [bool(openai_client), bool(groq_client), OLLAMA_INSTALLED],
}
availability_df = pd.DataFrame(availability)
print('\n### Provider Status ###')
print(availability_df)
print('\nNote: Ensure API keys are set in .env file or environment variables for OpenAI and Groq.')